In [1]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
print('Not connected to a GPU' if gpu_info.find('failed') >= 0 else gpu_info)

Sun Aug 30 16:00:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print(f'RAM disponible: {ram_gb:.1f} GB')
print('Not using a high-RAM runtime' if ram_gb < 20 else 'RAM alta activa')

RAM disponible: 54.8 GB
RAM alta activa


### 1. Carga de datos

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

esqueleto = pd.read_parquet('/content/drive/MyDrive/TFM/favorita_panel_corregido.parquet')
print(esqueleto.shape)

Mounted at /content/drive
(4804366, 22)


### 2. Preparación de variables

In [4]:
if 'type' in esqueleto.columns:
    esqueleto = esqueleto.rename(columns={'type': 'store_type'})

for col in ['store_nbr', 'item_nbr', 'city', 'state', 'store_type', 'cluster', 'family', 'class', 'perishable']:
    esqueleto[col] = esqueleto[col].astype(str)

### 3. Partición de datos

In [5]:
fecha_max = esqueleto['date'].max()
test_inicio = fecha_max - pd.Timedelta(days=60)
val_inicio = test_inicio - pd.Timedelta(days=60)

train = esqueleto[esqueleto['date'] < val_inicio]
val = esqueleto[(esqueleto['date'] >= val_inicio) & (esqueleto['date'] < test_inicio)]

print('Train:', train.shape, '| Val:', val.shape)

Train: (4322665, 22) | Val: (238860, 22)


### 4. Ponderación por volumen

In [6]:
volumen_por_serie = esqueleto.groupby(['store_nbr', 'item_nbr'])['unit_sales'].transform('mean')
esqueleto['peso_muestra'] = 1 / np.sqrt(volumen_por_serie + 0.1)
esqueleto['peso_muestra'] = esqueleto['peso_muestra'] / esqueleto['peso_muestra'].mean()

### 5. Instalación e importaciones

In [7]:
!pip install pytorch-forecasting pytorch-lightning lightning -q

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
import torch

print(torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 71.7 MB/s eta 0:00:00
True


### 6. Construcción del TimeSeriesDataSet de TFT

In [8]:
# Tres categorías de covariables (DeepAR solo admite target + conocidas a futuro)
# Estáticas: no cambian con el tiempo
# Conocidas a futuro (time_varying_known_reals): calculables de antemano para el horizonte de predicción
# Observadas solo en el pasado (time_varying_unknown_reals): junto con el target, solo disponibles en el encoder.
# Aquí reingresa dcoilwtico

max_encoder_length = 90   #mismo contexto usado para fijar el umbral de 106 días
max_prediction_length = 30  #mismo horizonte que DeepAR

training_cutoff = train['time_idx'].max()
CUANTILES = [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]  #mismos 7 cuantiles que DeepAR

training = TimeSeriesDataSet(
    esqueleto[esqueleto.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="unit_sales",
    group_ids=["store_nbr", "item_nbr"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    static_categoricals=["city", "state", "store_type", "cluster", "family", "class", "perishable"],

    time_varying_known_reals=["time_idx", "year", "month", "day_of_week", "is_weekend",
                               "es_feriado", "onpromotion", "edad"],

    time_varying_unknown_reals=["unit_sales", "dcoilwtico"],  #petróleo incorporado

    #mismo tratamiento de ceros ya validado para DeepAR (log1p, no softplus)
    target_normalizer=GroupNormalizer(groups=["store_nbr", "item_nbr"], transformation="log1p"),

    weight="peso_muestra",

    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=False,
)

validation = TimeSeriesDataSet.from_dataset(
    training, esqueleto, min_prediction_idx=training_cutoff + 1, stop_randomization=True
)

/usr/local/lib/python3.13/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 88 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__store_nbr': '1', '__group_id__item_nbr': '1428779'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2002136'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2027777'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2027827'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053590'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053610'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053614'}, {'__group_id__store_nbr': '10', '__group_id__item_nbr': '2033805'}, {'__group_id__store_nbr': '10', '__group_id__item_nbr': '2053590'}, {'__group_id__store_nbr': '1

In [9]:
# Prueba de parámetros para las nuevas configuraciones — no entrena nada, solo instancia y cuenta
for hs, ahs, hcs, nombre in [(30, 1, 15, 't1/t3 nuevas'), (64, 4, 32, 't2 nueva')]:
    modelo_prueba = TemporalFusionTransformer.from_dataset(
        training,
        hidden_size=hs,
        attention_head_size=ahs,
        hidden_continuous_size=hcs,
        loss=QuantileLoss(quantiles=CUANTILES),
    )
    print(f'{nombre} (hidden_size={hs}): {modelo_prueba.size() / 1e3:.1f}k parámetros')
    del modelo_prueba

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


t1/t3 nuevas (hidden_size=30): 91.5k parámetros
t2 nueva (hidden_size=64): 373.1k parámetros


### 7. Verificación de configuraciones

In [10]:
print('--- Estáticas ---')
print(training.static_categoricals)
print('--- Conocidas a futuro ---')
print(training.time_varying_known_reals)
print('--- Observadas solo en el pasado (incluye el target) ---')
print(training.time_varying_unknown_reals)
assert 'dcoilwtico' in training.time_varying_unknown_reals, 'El petróleo debe estar aquí, no en conocidas a futuro'
assert 'dcoilwtico' not in training.time_varying_known_reals, 'El petróleo NO debe estar en conocidas a futuro'
print('\nVerificación de petróleo: OK')

--- Estáticas ---
['city', 'state', 'store_type', 'cluster', 'family', 'class', 'perishable']
--- Conocidas a futuro ---
['time_idx', 'year', 'month', 'day_of_week', 'is_weekend', 'es_feriado', 'onpromotion', 'edad']
--- Observadas solo en el pasado (incluye el target) ---
['unit_sales', 'dcoilwtico']

Verificación de petróleo: OK


### 8. Dataloaders

In [11]:
batch_size = 128
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=0)

### 9. Prueba rápida de instanciación

In [12]:
#para confirmar que el modelo se arma sin error y que la pérdida de cuantiles y la
#capa de atención quedan configuradas correctamente.
# TemporalFusionTransformer de pytorch-forecasting implementa por diseño una sola capa de atención
# interpretable -- no hay parámetro para apilar más de una, así queda satisfecho por uso de la clase estándar de la librería
tft_prueba = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=16,             # valor pequeño solo para la prueba
    attention_head_size=1,
    dropout=0.1,
    hidden_continuous_size=8,
    loss=QuantileLoss(quantiles=CUANTILES),
    log_interval=0,
)

print(f'Número de parámetros: {tft_prueba.size() / 1e3:.1f}k')

trainer_prueba = pl.Trainer(
    max_epochs=1, accelerator="auto", enable_model_summary=True,
    limit_train_batches=5, limit_val_batches=2,  # solo un par de lotes, no es entrenamiento real
    logger=False,
)
trainer_prueba.fit(tft_prueba, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
print('\nPrueba de instanciación y forward/backward pass: OK — configuración lista para entrenar en serio en TFT_d2')

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Número de parámetros: 30.4k


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │    275 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    224 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  2.8 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  7.4 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  5.9 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  1.1 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    119 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 30.4 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 30.4 K                                                                                               
Total estimated model params size (MB): 0.122                                                                      
Modules in train mode: 526                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

INFO: `Trainer.fit` stopped: `max_epochs=1` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.



Prueba de instanciación y forward/backward pass: OK — configuración lista para entrenar en serio en TFT_d2
